In [1]:
import numpy as np
from graphlib import TopologicalSorter, CycleError

In [ ]:
class Resource:

    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"{self.name}"


class Process:

    def __init__(self, name: str):
        self.inputs: dict[Resource, float] = {}
        self.outputs: dict[Resource, float] = {}
        self.name = name

    def add_input(self, resource, qty):
        if qty >= 0:
            self.inputs[resource] = qty
        else:
            raise ValueError(f"Количество не может быть отрицательным: {qty}")

    def add_output(self, resource, qty):
        if qty >= 0:
            self.outputs[resource] = qty
        else:
            raise ValueError(f"Количество не может быть отрицательным: {qty}")

    def __repr__(self):
        ins = [f"{r.name} {q}" for r, q in self.inputs.items()]
        outs = [f"{r.name} {q}" for r, q in self.outputs.items()]

        rows = max(len(ins), len(outs))
        col_width = max((len(s) for s in ins), default=0)

        lines = [f"{self.name!r}"]
        for i in range(rows):
            left = ins[i] if i < len(ins) else ""
            right = outs[i] if i < len(outs) else ""
            lines.append(f"  {left:<{col_width}}  |  {right}")

        return "\n".join(lines)


class ProductionChain:

    def __init__(self, name):
        self.name = name
        self.processes: list[Process] = []

    def add_process(self, process):
        self.processes.append(process)

    def resources(self):
        inputs = set()
        outputs = set()

        for process in self.processes:
            inputs.update(process.inputs.keys())
            outputs.update(process.outputs.keys())

        intermediate_products = inputs & outputs
        raw_materials = inputs - outputs
        final_products = outputs - inputs

        return raw_materials, intermediate_products, final_products

    def edges(self):
        result = []
        for producer in self.processes:
            for resource in producer.outputs:
                for consumer in self.processes:
                    if resource in consumer.inputs:
                        result.append((producer.name, resource.name, consumer.name))
        return result


    def dependencies(self) -> dict[Process, set[Process]]:
        deps = {p: set() for p in self.processes}
        for producer, resource, consumer in self.edges():
            if producer is not consumer:
                deps[consumer].add(producer)
        return deps

    def execution_order(self) -> list[Process]:
        try:
            return list(TopologicalSorter(self.dependencies()).static_order())
        except CycleError as e:
            cycle = e.args[1]
            raise ValueError(
                "В цепочке есть цикл: " + " -> ".join(p.name for p in cycle)
            ) from None

    def __repr__(self):
        raw_materials, intermediate_products, final_products = self.resources()

        return (
            f"ProductionChain(name={self.name!r})\n"
        )
        

In [3]:
crude_oil = Resource("crude oil")
gasoline = Resource("gasoline")
diesel = Resource("diesel")

ethanol = Resource("ethanol")
premium_gasoline = Resource("premium_gasoline")


distillation = Process('distilation')
distillation.add_input(crude_oil, 1)
distillation.add_output(gasoline, 0.3)
distillation.add_output(diesel, 0.2)

blending = Process('blending')
blending.add_input(ethanol, 0.1)
blending.add_input(gasoline, 1)
blending.add_output(premium_gasoline, 1)

In [4]:
refinery = ProductionChain("refinery")
refinery.add_process(distillation)
refinery.add_process(blending)

In [5]:
refinery.processes

['distilation'
   crude oil 1  |  gasoline 0.3
                |  diesel 0.2,
 'blending'
   ethanol 0.1  |  premium_gasoline 1
   gasoline 1   |  ]

In [6]:
demands = {gasoline: 5, diesel: 4}

def compute_process_runs(chain, demands):
    process_runs = {}
    for process in chain.processes:          # process — локальная, в globals() не попадает
        runs = {}
        for output, qty_per_run in process.outputs.items():
            if output in demands:
                runs[output] = demands[output] / qty_per_run
        process_runs[process.name] = runs
    return process_runs

process_runs = compute_process_runs(refinery, demands)

In [7]:
refinery

ProductionChain(name='refinery')
  Сырьё: {crude oil, ethanol}
  Промежуточные продукты: {gasoline}
  Конечные продукты: {diesel, premium_gasoline}

In [8]:
economy = []

for name, obj in list(globals().items()):
    if isinstance(obj, Process):
        economy.append(obj)

economy

['distilation'
   crude oil 1  |  gasoline 0.3
                |  diesel 0.2,
 'blending'
   ethanol 0.1  |  premium_gasoline 1
   gasoline 1   |  ]

In [10]:
blending.outputs

{premium_gasoline: 1}

In [13]:
def process_dfs(economy, demand, visited = None):

    if visited is None:
        visited = set()

    for process in economy:
        if process in visited:
            continue

        if demand in process.outputs:
            visited.add(process)

            for input in process.inputs: 
                process_dfs(economy, input)

    return visited


In [14]:
process_dfs(economy, diesel)

{'distilation'
   crude oil 1  |  gasoline 0.3
                |  diesel 0.2}